In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import pickle
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
base_path = "/content/drive/MyDrive/TicketMind/"

intent_model_path = base_path + "model_final"
intent_tokenizer = AutoTokenizer.from_pretrained(intent_model_path)
intent_model = AutoModelForSequenceClassification.from_pretrained(intent_model_path)
intent_model.to("cuda")
intent_model.eval()

with open(intent_model_path + "/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0,
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
train_df = pd.read_csv(base_path + "data/train_data.csv")
instruction_embeddings = np.load(base_path + "data/instruction_embeddings.npy")

Loading weights:   0%|          | 0/104 [00:04<?, ?it/s]

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
def predict_intent(text: str, top_k: int = 1):
    inputs = intent_tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = intent_model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)

    predicted_id = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][predicted_id].item()
    predicted_intent = label_encoder.inverse_transform([predicted_id])[0]

    return {
        "intent": predicted_intent,
        "confidence": round(confidence, 4),
    }

In [5]:
def get_sentiment(text: str):
    result = sentiment_analyzer(text)[0]
    return {
        "sentiment": result["label"].lower(),
        "confidence": round(result["score"], 4),
    }

In [6]:
def find_similar_response(new_message: str, predicted_intent: str, top_k: int = 3):
    new_embedding = embedding_model.encode([new_message])
    similarities = cosine_similarity(new_embedding, instruction_embeddings)[0]

    sorted_indices = similarities.argsort()[::-1]

    matched_results = []
    for idx in sorted_indices:
        row = train_df.iloc[idx]
        if row["intent"] == predicted_intent:
            matched_results.append({
                "matched_instruction": row["instruction"],
                "suggested_response": row["response"],
                "similarity_score": round(float(similarities[idx]), 4),
            })
        if len(matched_results) >= top_k:
            break

    if not matched_results:
        top_indices = sorted_indices[:top_k]
        for idx in top_indices:
            row = train_df.iloc[idx]
            matched_results.append({
                "matched_instruction": row["instruction"],
                "suggested_response": row["response"],
                "similarity_score": round(float(similarities[idx]), 4),
            })

    return matched_results

In [17]:
def process_customer_message(message: str, intent_confidence_threshold: float = 0.5):
    intent_result = predict_intent(message)
    sentiment_result = get_sentiment(message)
    low_confidence_intent = intent_result["confidence"] < intent_confidence_threshold

    if low_confidence_intent:
        return {
            "original_message": message,
            "predicted_intent": "unclear / general_comment",
            "intent_confidence": intent_result["confidence"],
            "sentiment": sentiment_result["sentiment"],
            "sentiment_confidence": sentiment_result["confidence"],
            "priority": "low",
            "note": "Message does not clearly match a known support intent (e.g. a thank-you or general comment). No automatic response suggested - route to a general acknowledgment or human review.",
            "suggested_response": None,
            "similarity_score": None,
        }

    similar_results = find_similar_response(
        message, predicted_intent=intent_result["intent"], top_k=1,
    )
    best_match = similar_results[0]

    if sentiment_result["sentiment"] == "negative" and sentiment_result["confidence"] > 0.8:
        priority = "high"
        note = "Customer appears clearly upset - recommend reviewing the response before sending and prioritizing a quick reply"
    elif sentiment_result["sentiment"] == "negative":
        priority = "medium"
        note = "Customer seems dissatisfied - please review the suggested response before sending"
    else:
        priority = "normal"
        note = "The suggested response can be sent directly after a quick review"

    return {
        "original_message": message,
        "predicted_intent": intent_result["intent"],
        "intent_confidence": intent_result["confidence"],
        "sentiment": sentiment_result["sentiment"],
        "sentiment_confidence": sentiment_result["confidence"],
        "priority": priority,
        "note": note,
        "suggested_response": best_match["suggested_response"],
        "similarity_score": best_match["similarity_score"],
    }

In [18]:
test_cases = [
    "I want to cancel my order right now, this is ridiculous!",
    "How can I track my refund status?",
    "Thank you so much, everything was perfect!",
]

for msg in test_cases:
    result = process_customer_message(msg)
    print("=" * 60)
    print(f"Message: {result['original_message']}")
    print(f"Predicted Intent: {result['predicted_intent']} (confidence: {result['intent_confidence']*100:.1f}%)")
    print(f"Sentiment: {result['sentiment']} (confidence: {result['sentiment_confidence']*100:.1f}%)")
    print(f"Priority: {result['priority']}")
    print(f"Note: {result['note']}")
    print(f"Suggested Response: {str(result['suggested_response'])[:150]}...")
    print()

Message: I want to cancel my order right now, this is ridiculous!
Predicted Intent: cancel_order (confidence: 99.7%)
Sentiment: negative (confidence: 93.0%)
Priority: high
Note: Customer appears clearly upset - recommend reviewing the response before sending and prioritizing a quick reply
Suggested Response: I pick up what you're putting down, your need to cancel your order. Let's make this process as smooth as possible. Here's what you need to do:

1. Log...

Message: How can I track my refund status?
Predicted Intent: track_refund (confidence: 99.9%)
Sentiment: neutral (confidence: 87.9%)
Priority: normal
Note: The suggested response can be sent directly after a quick review
Suggested Response: I certainly recognize your need to check the status of your refund. Let me guide you through the process. To see the status of your refund, you can vi...

Message: Thank you so much, everything was perfect!
Predicted Intent: unclear / general_comment (confidence: 28.4%)
Sentiment: positive (co